## LSA(잠재 의미 분석)
- 문서 안에서 단어 사이의 잠재적인 의미 구조를 추출하는 기법
- TF-IDF 방식은 단어 간의 의미적 유사성을 반영X
- TF-IDF에서 SVD 분해를 하여 단어 간의 의미를 파악
- TF-IDF에서 차원 축소(PCA, t-SNE과 같은 축소 기법 사용)하여 과계성을 확인
- LSA 효과
    - 백터 공간의 차원을 줄여서 계싼 효율 증가(차원 축소)
    - '영화', '필름' 비슷한 문맥의 단어를 가까운 백터로 이동(의미 유추)
    - 문서들을 주제별로 분류 기능(토픽 분석)

- TruncatedSVD (차원 축소 모델)
    - 절단된 특이값의 분해
    - 고차원 희소 행렬 (값이 0인 행렬)을 낮은 차원으로 압축하여 데이터 구조적 의미를 유지
    - 자연어 처리, 추천 시스템, 의미 분석, 잠재적인 토픽 분석 주로 사용

    - TF-IDF 행렬은 우선은 고차원 -> 저차원
    - 0으로 이루어진 희소행렬들을 구조적인 의미를 유지하면서값들을 부여할 수 있다.
        - 같은 토픽의 문서는 같은 벡터 공간에서 가깝게 위치 -> 유사도 기반 자연어처리에 활용

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
import pandas as pd
from konlpy.tag import Okt

In [2]:
# 토큰화 함수를 정의 -> pos 필터
okt = Okt()

def tokenize(text):
    result = [word for word, pos in okt.pos(text) if pos in ['Noun', 'Adjective', 'Verb']]
    return result
    

In [3]:
docs = [
    '이 영화 정말 재미있었다',
    '매우 연기가 뛰어나다',
    '이 영화 별로다',
    '지루한 영화는 보기 어렵다',
    '정말 훌륭한 연기였다',
    '연기가 별로라서 지루했다'
]

In [4]:
# TF-IDF 백터화
tfidf = TfidfVectorizer(
    tokenizer=tokenize,
    ngram_range=(1, 1),
    min_df = 1,
    max_df = 0.8,
    sublinear_tf = True,
    lowercase=False
)

In [5]:
X_tfidf = tfidf.fit_transform(docs)

c:\Users\abohv\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [6]:
X_tfidf.shape

(6, 14)

In [8]:
# SVD를 이용한 차원 축소 (LSA 적용)
lsa = TruncatedSVD(n_oversamples=2, random_state=42)
X_lsa = lsa.fit_transform(X_tfidf)

In [11]:
df_lsa = pd.DataFrame(X_lsa, columns=['topic1', 'topic2'])
df_lsa['doucment'] = docs

In [12]:
df_lsa

,topic1,topic2,doucment
0,0.693629,-0.340174,이 영화 정말 재미있었다
1,0.205175,0.675296,매우 연기가 뛰어나다
2,0.798874,-0.278481,이 영화 별로다
3,0.374011,-0.379133,지루한 영화는 보기 어렵다
4,0.394059,0.481138,정말 훌륭한 연기였다
5,0.546162,0.498158,연기가 별로라서 지루했다


In [15]:
terms = tfidf.get_feature_names_out()
components = lsa.components_

df_terms = pd.DataFrame(components.T, index = terms,
                        columns= ['topic1', 'topic2'])

df_terms.sort_values('topic1')

,topic1,topic2
뛰어나다,0.061571,0.335831
매우,0.061571,0.335831
어렵다,0.123537,-0.158865
지루한,0.123537,-0.158865
보기,0.123537,-0.158865
였다,0.130528,0.213372
훌륭한,0.130528,0.213372
재미있었다,0.224303,-0.160163
지루했다,0.225018,0.267923
연기,0.288775,0.565706
